In [7]:
import sys
import site

USER_SITE = site.getusersitepackages()

if USER_SITE not in sys.path:
    sys.path.insert(0, USER_SITE)

import peft

print("PEFT loaded:", peft.__version__)

PEFT loaded: 0.19.1


In [8]:
from datasets import load_dataset

dataset = load_dataset("nyu-mll/glue", "mrpc")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


In [9]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print("Tokenization done")

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Tokenization done


In [11]:
from peft import LoraConfig, get_peft_model, TaskType

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

model = get_peft_model(model, lora_config)
model.to("cuda")

model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [13]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = (preds == labels).mean()
    f1 = f1_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1
    }

In [17]:
from pathlib import Path

BASE = Path.home() / "MyWork" / "LoRA_Reproduction"

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(BASE / "nlu" / "mrpc" / "output"),

    learning_rate=5e-4,
    num_train_epochs=10,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    report_to="none"
)

In [20]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

print("Trainer ready")

Trainer ready


In [21]:
model.config.use_cache = False
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.677689,0.398111,0.825980,0.872531
2,0.287546,0.398662,0.857843,0.897887
3,0.322225,0.362698,0.848039,0.885185
4,0.298870,0.467611,0.838235,0.890365
5,0.298651,0.420839,0.855392,0.896309
6,0.251764,0.403341,0.852941,0.892086
7,0.176185,0.550808,0.860294,0.899824
8,0.095591,0.481992,0.857843,0.901024
9,0.293406,0.625246,0.855392,0.898451
10,0.076986,0.625210,0.867647,0.904594


TrainOutput(global_step=2300, training_loss=0.2690090374842934, metrics={'train_runtime': 132.5773, 'train_samples_per_second': 276.669, 'train_steps_per_second': 17.348, 'total_flos': 2437716563681280.0, 'train_loss': 0.2690090374842934, 'epoch': 10.0})

In [7]:
from pathlib import Path
import json

BASE = Path.home() / "MyWork" / "LoRA_Reproduction"
LOGS_DIR = BASE / "results" / "logs"

results = {
    "task": "MRPC",
    "accuracy": 0.8676,
    "f1": 0.9046
}

with open(LOGS_DIR / "mrpc_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("MRPC saved!")

MRPC saved!
